# 13 · ShakerMaker vs SW4

Cross-validate the FK engine against a full **SW4** finite-difference run on the *same* model: a 4-layer crust, 100 FFSP point sources, and 2 surface stations. Both solvers are driven from one ShakerMaker export (`data/model_summary.h5`), so source and geometry are identical — any difference is method, not setup.

Run from this `notebooks/` folder (paths point to `../data`). Needs `obspy` for the band-pass filter.

In [1]:
import ast
from pathlib import Path
import h5py
import numpy as np
import matplotlib.pyplot as plt

from shakermaker import shakermaker
from shakermaker.crustmodel import CrustModel
from shakermaker.pointsource import PointSource
from shakermaker.faultsource import FaultSource
from shakermaker.station import Station
from shakermaker.stationlist import StationList
from shakermaker.stf_extensions import Discrete

DATA = Path('../data')
PACKAGE_H5 = DATA / 'model_summary.h5'
FMIN, FMAX = 0.25, 15.0   # comparison band (Hz)


    ▄████████     ▄█    █▄     ▄████████     ▄█   ▄█▄    ▄████████    ▄████████
    ███    ███   ███    ███    ███    ███    ███ ▄███▀   ███    ███   ███    ███
    ███    █▀    ███    ███    ███    ███    ███▐██▀     ███    █▀    ███    ███
    ███         ▄███▄▄▄▄███▄▄  ███    ███   ▄█████▀     ▄███▄▄▄      ▄███▄▄▄▄██▀
  ▀███████████  ▀███▀▀▀▀███▀  ▀███████████  ▀█████▄    ▀▀███▀▀▀     ▀▀███▀▀▀▀▀
           ███   ███    ███    ███    ███    ███▐██▄     ███    █▄   ▀███████████
     ▄█    ███   ███    ███    ███    ███    ███ ▀███▄   ███    ███   ███    ███
   ▄████████▀    ███    █▀     ███    █▀     ███   ▀█▀   ██████████   ███    ███
                                                                      ███    ███

         ▄▄▄▄███▄▄▄▄      ▄████████    ▄█   ▄█▄    ▄████████    ▄████████
       ▄██▀▀▀███▀▀▀██▄   ███    ███    ███ ▄███▀   ███    ███   ███    ███
       ███   ███   ███   ███    ███    ███▐██▀     ███    █▀    ███    ███
       ███   ███   ███   ███    ███   ▄█████▀  

## Step 1 — Rebuild the model from the SW4 package

### The crust

In [2]:
with h5py.File(PACKAGE_H5, 'r') as f:
    g = f['crust']
    thickness, vp, vs = g['thickness_km'][:], g['vp_km_s'][:], g['vs_km_s'][:]
    rho, qp, qs = g['rho_g_cm3'][:], g['qp'][:], g['qs'][:]

crust = CrustModel(len(thickness))
for i in range(len(thickness)):
    crust.add_layer(thickness[i], vp[i], vs[i], rho[i], qp[i], qs[i])
print(crust)
crust.plot()

Crust Model with 4 layers.
 Layer  | Depth  | Thick  | Vp     | Vs     | rho    | Qa     | Qb
       1|    0.00|    0.20|    1.32|    0.75|    2.40|  1000.0|  1000.0
       2|    0.20|    0.80|    2.75|    1.57|    2.50|  1000.0|  1000.0
       3|    1.00|   14.50|    5.50|    3.14|    2.50|  1000.0|  1000.0
       4|   15.50|    0.00|    7.00|    4.00|    2.67|  1000.0|  1000.0



<Figure size 600x400 with 1 Axes>

### The 100 sources (Discrete slip-rate STFs)

In [3]:
with h5py.File(PACKAGE_H5, 'r') as f:
    s = f['sources']
    x, y, z = s['x_km'][:], s['y_km'][:], s['z_km'][:]
    strike, dip, rake = s['strike_deg'][:], s['dip_deg'][:], s['rake_deg'][:]
    tt, dt_all, npts_all = s['trigger_time_s'][:], s['dt'][:], s['npts'][:]
    offsets, values = s['data_offsets'][:], s['data_values'][:]

sources = []
for i in range(len(x)):
    dt, npts, off = float(dt_all[i]), int(npts_all[i]), int(offsets[i])
    stf = Discrete(values[off:off + npts], np.arange(npts) * dt)
    stf.dt = dt
    sources.append(PointSource([x[i], y[i], z[i]],
                               [strike[i], dip[i], rake[i]], tt=tt[i], stf=stf))

fault = FaultSource(sources, metadata={'name': 'from_sw4_package'})
print('sources:', fault.nsources)

sources: 100


In [4]:
# the 100 slip-rate functions, each at its own rupture time tt (no legend)
plt.figure(figsize=(10, 4))
for i in range(len(x)):
    dt, npts, off = float(dt_all[i]), int(npts_all[i]), int(offsets[i])
    t = float(tt[i]) + np.arange(npts) * dt
    plt.plot(t, values[off:off + npts], linewidth=1)
plt.xlabel('Time [s]'); plt.ylabel('Source time function')
plt.title('Source time functions (100 sources)')
plt.grid(True); plt.tight_layout(); plt.show()

### The stations

In [5]:
with h5py.File(PACKAGE_H5, 'r') as f:
    g = f['stations']
    xyz, internal, meta_raw = g['xyz_km'][:], g['internal'][:], g['metadata'][:]

stations_list = []
for i in range(len(xyz)):
    txt = meta_raw[i].decode() if isinstance(meta_raw[i], bytes) else meta_raw[i]
    try:
        meta = ast.literal_eval(txt)
    except Exception:
        meta = {'name': 'sta_%d' % i}
    stations_list.append(Station(xyz[i], internal=bool(internal[i]), metadata=meta))

stations = StationList(stations_list, metadata={'name': 'from_sw4_package'})

plt.figure(figsize=(7, 7))
for i, sta in enumerate(stations_list):
    yy, xx, zz = sta.x
    plt.scatter(xx, yy)
    plt.text(xx, yy, '%d: %s' % (i, sta.metadata.get('name', '')), fontsize=9)
plt.xlabel('X [km]'); plt.ylabel('Y [km]'); plt.axis('equal'); plt.grid(True); plt.show()

## Step 2 — The FK engine result

The FK run is heavy (`nfft=32768`, 100 sources x 2 stations = 200 core calls, about 2 min each), so it is run by the
**script** under MPI, which also checks the agreement with SW4 and saves both station responses:

```bash
cd ..; mpiexec -n 16 python shaker_vs_sw4.py      # ~30 min with 16 processes, ~80 min with 6
```

This notebook reads `../_output/shakermaker_sf0000{1,2}.npz`, written by that run.

In [6]:
model = shakermaker.ShakerMaker(crust, fault, stations)
model.check_parameters(dt=0.0025, nfft=8192 * 4, dk=0.2, tb=800, tmax=60)

 ShakerMaker . PARAMETER CHECK            you set:  dt + tmax
 YOU CHOSE     dt = 0.0025 s        tmax = 60 s
 GEOMETRY      r_max 29.4 km . src-rcv sep 2.9-12.9 km . Vs_min 750 m/s
               Vp_max 7000 m/s . V_Ray 0.69 km/s
               physical signal window  t = [2.7, 60.3] s  (lasts 57.6 s)
----------------------------------------------------------------------
 dt = 0.0025 s   sets your FREQUENCY BAND                       [info]
   f_Nyq         = 1/(2*dt)                   = 200 Hz       [fk.f:72]
   f_max usable  = (1-taper)*f_Nyq, taper=0.9 = 20 Hz        [fk.f:74]
   above 20 Hz   fades out smoothly, fully gone by 200 Hz    [fk.f:169]
   lambda_min    = Vs_min/f_max               = 750/20 = 37.5 m
     element size (N=10 pts/wavelength), for YOUR mesh:
        dx <= lambda_min/N    = 37.5/10 = 3.75 m
        dt <= C*dx/Vp_surf    = 1*3.75/1320 = 0.0028 s
        (Vp_surf = Vp of the soft Vs_min layer, NOT Vp_max:
         the 3.8 m elements live in the soft surface lay

{'passed': True,
 'recommended': {'dk': 0.4, 'tb': 800, 'nfft': 32768, 'tmax': 60.3}}

In [7]:
out = Path('../_output')
sm = {name: np.load(out / f'shakermaker_{name}.npz') for name in ('sf00001', 'sf00002')}
z_sm, e_sm, n_sm, t_sm = (sm['sf00001'][k] for k in ('z', 'e', 'n', 't'))
print({name: d['t'].size for name, d in sm.items()})

{'sf00001': 24001, 'sf00002': 24001}


In [8]:
fig, ax = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for k, (d, lab) in enumerate([(z_sm, r'$u_z$'), (e_sm, r'$u_e$'), (n_sm, r'$u_n$')]):
    ax[k].plot(t_sm, d, 'r', lw=1); ax[k].set_xlim(0, 15)
    ax[k].set_ylabel(lab); ax[k].grid(True)
ax[2].set_xlabel('Time [s]'); fig.suptitle('ShakerMaker velocity, station sf00001 (unfiltered)'); plt.show()

## Step 3 — Read and band-pass the SW4 output

SW4 columns `(time, x, y, z)` map to `u_z=z, u_e=y, u_n=x`. Both solutions are band-passed with ObsPy to the same
band so they are compared on equal footing.

In [9]:
from obspy import Trace

def read_sw4_station(name):
    a = np.loadtxt(DATA / ('%s.txt' % name), skiprows=13)
    return a[:, 0], a[:, 3], a[:, 2], a[:, 1]   # t, u_z, u_e, u_n

def bandpass(sig, dt):
    tr = Trace(data=sig.astype(np.float32)); tr.stats.delta = dt
    tr.filter('bandpass', freqmin=FMIN, freqmax=FMAX, corners=4, zerophase=True)
    return tr.data

t_s, z_s, e_s, n_s = read_sw4_station('sf00001')
dt_s = t_s[1] - t_s[0]
z_f, e_f, n_f = bandpass(z_s, dt_s), bandpass(e_s, dt_s), bandpass(n_s, dt_s)

In [10]:
fig, ax = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for k, (d, lab) in enumerate([(z_f, r'$u_z$'), (e_f, r'$u_e$'), (n_f, r'$u_n$')]):
    ax[k].plot(t_s, d); ax[k].set_xlim(0, 15); ax[k].set_ylabel(lab); ax[k].grid(True)
ax[2].set_xlabel('Time [s]'); fig.suptitle('SW4 velocity (filtered)'); plt.show()

## Step 4 — Overlay and compare

SW4 dashed in the background, ShakerMaker on top, both in the same band. The script fails if the correlation or the
peak ratio leaves its limits.

In [11]:
dt_sm = t_sm[1] - t_sm[0]
tc = np.arange(0.0, 15.0, dt_s)
for name in ('sf00001', 'sf00002'):
    t_s, z_s, e_s, n_s = read_sw4_station(name)
    sw = [bandpass(c, dt_s) for c in (z_s, e_s, n_s)]
    fk = [bandpass(sm[name][c], dt_sm) for c in ('z', 'e', 'n')]
    fig, ax = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
    for k, lab in enumerate([r'$u_z$', r'$u_e$', r'$u_n$']):
        a, b = np.interp(tc, t_sm, fk[k]), np.interp(tc, t_s, sw[k])
        cc = a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
        ax[k].plot(t_s, sw[k], '--', color='#7f9fc6', lw=1.2, label='SW4')
        ax[k].plot(t_sm, fk[k], color='#c0392b', lw=0.9, label='ShakerMaker')
        ax[k].text(0.99, 0.85, f'cc {cc:.3f}, peak ratio {np.abs(a).max() / np.abs(b).max():.2f}',
                   transform=ax[k].transAxes, ha='right', fontsize=8)
        ax[k].set_xlim(0, 15); ax[k].set_ylabel(lab); ax[k].grid(True)
    ax[0].legend(loc='upper left'); ax[2].set_xlabel('Time [s]')
    fig.suptitle(f'ShakerMaker vs SW4, station {name}, {FMIN:g}-{FMAX:g} Hz'); plt.tight_layout(); plt.show()